# 02 — Model Training & Evaluation
**Smart Health Risk Predictor — ML CCP**

Trains and compares **five classifiers** per disease:

1. Logistic Regression
2. K-Nearest Neighbors
3. Decision Tree
4. Random Forest
5. SVM (RBF)

Evaluates with accuracy / precision / recall / F1 / confusion matrix and
**5-fold cross-validation**, then runs **GridSearchCV** on the best
algorithm and pickles `model.pkl`, `scaler.pkl`, `encoder.pkl`.

In [ ]:
import json, joblib, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors  import KNeighborsClassifier
from sklearn.tree       import DecisionTreeClassifier
from sklearn.ensemble   import RandomForestClassifier
from sklearn.svm        import SVC
from sklearn.metrics    import (accuracy_score, precision_score, recall_score,
                                f1_score, confusion_matrix, classification_report)
from sklearn.model_selection import cross_val_score, GridSearchCV

DATA = Path('../dataset/clean')
MODEL_DIR = Path('../model'); MODEL_DIR.mkdir(exist_ok=True)
prep = joblib.load(DATA/'prepared.joblib')
splits, scalers, encoders, DISEASES = prep['splits'], prep['scalers'], prep['encoders'], prep['diseases']

MODELS = {
    'LogReg': LogisticRegression(max_iter=1000),
    'KNN':    KNeighborsClassifier(),
    'DTree':  DecisionTreeClassifier(random_state=42),
    'RF':     RandomForestClassifier(n_estimators=200, random_state=42),
    'SVM':    SVC(probability=True),
}
list(splits)

## 1. Train & Compare

In [ ]:
def evaluate(y_true, y_pred, avg='binary' if True else 'macro'):
    try:
        return {
            'accuracy':  accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
            'f1':        f1_score(y_true, y_pred, average='macro', zero_division=0),
        }
    except Exception as e:
        return {'error': str(e)}

all_results = {}
fitted = {}
for d, (X_tr, X_te, y_tr, y_te, feats) in splits.items():
    print(f"\n=== {d.upper()} ===")
    res = {}
    fitted[d] = {}
    for n, m in MODELS.items():
        m.fit(X_tr, y_tr)
        pred = m.predict(X_te)
        scores = evaluate(y_te, pred)
        cv = cross_val_score(m, X_tr, y_tr, cv=5).mean()
        scores['cv_accuracy'] = float(cv)
        scores['confusion_matrix'] = confusion_matrix(y_te, pred).tolist()
        res[n] = {k: (float(v) if isinstance(v,(int,float,np.floating)) else v) for k,v in scores.items()}
        fitted[d][n] = m
        print(f"  {n:7s} acc={scores['accuracy']:.4f}  cv={cv:.4f}")
    all_results[d] = res

## 2. Comparison Table

In [ ]:
rows = []
for d, info in all_results.items():
    for m, r in info.items():
        rows.append({'disease':d,'model':m,'accuracy':r['accuracy'],
                     'precision':r['precision'],'recall':r['recall'],
                     'f1':r['f1'],'cv':r['cv_accuracy']})
cmp = pd.DataFrame(rows)
cmp.pivot(index='model', columns='disease', values='accuracy').plot(kind='bar', figsize=(8,4))
plt.title('Accuracy per model per disease'); plt.ylabel('accuracy'); plt.tight_layout(); plt.show()
cmp

## 3. Pick best model per disease & GridSearchCV

In [ ]:
GRIDS = {
    'LogReg': {'C':[0.1,1,10]},
    'KNN':    {'n_neighbors':[3,5,7,9]},
    'DTree':  {'max_depth':[None,5,10,20]},
    'RF':     {'n_estimators':[100,200], 'max_depth':[None,10,20]},
    'SVM':    {'C':[0.5,1,5], 'kernel':['rbf','linear']},
}

best_per_disease = {}
for d, info in all_results.items():
    best_name = max(info, key=lambda k: info[k]['accuracy'])
    print(f"{d}: best = {best_name} (acc={info[best_name]['accuracy']:.4f})")
    X_tr, X_te, y_tr, y_te, feats = splits[d]
    gs = GridSearchCV(MODELS[best_name].__class__(**({'probability':True} if best_name=='SVM' else {})
                                                   | ({'max_iter':1000} if best_name=='LogReg' else {})
                                                   | ({'random_state':42} if best_name in ('RF','DTree') else {})),
                      GRIDS[best_name], cv=5, n_jobs=1)
    gs.fit(X_tr, y_tr)
    tuned = gs.best_estimator_
    pred  = tuned.predict(X_te)
    final = evaluate(y_te, pred); final['cv_accuracy'] = float(gs.best_score_)
    print(f"   tuned acc={final['accuracy']:.4f}  best params={gs.best_params_}")
    best_per_disease[d] = {'name':best_name,'model':tuned,'metrics':final,
                           'best_params':gs.best_params_,'features':feats}

## 4. Confusion Matrices

In [ ]:
import seaborn as sns
fig, axes = plt.subplots(1, len(best_per_disease), figsize=(4*len(best_per_disease),4))
if len(best_per_disease)==1: axes=[axes]
for ax,(d,info) in zip(axes, best_per_disease.items()):
    X_tr, X_te, y_tr, y_te, feats = splits[d]
    cm = confusion_matrix(y_te, info['model'].predict(X_te))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f"{d} - {info['name']}")
plt.tight_layout(); plt.show()

## 5. Save pickles (model / scaler / encoder)

In [ ]:
for d, info in best_per_disease.items():
    joblib.dump(info['model'], MODEL_DIR/f'{d}_model.pkl')
    joblib.dump(scalers[d],    MODEL_DIR/f'{d}_scaler.pkl')
    enc = dict(encoders[d]); enc['__features__'] = info['features']
    joblib.dump(enc, MODEL_DIR/f'{d}_encoder.pkl')
    print(f"saved {d}_model/scaler/encoder.pkl")

summary = {d:{'best':info['name'],'metrics':info['metrics'],
              'best_params':info['best_params'],'features':info['features'],
              'all_models': all_results[d]} for d,info in best_per_disease.items()}
(MODEL_DIR/'metrics.json').write_text(json.dumps(summary, indent=2, default=str))
print('metrics.json written')

## 6. Load & Predict (demo)

In [ ]:
bundle_disease = list(best_per_disease)[0]
model  = joblib.load(MODEL_DIR/f'{bundle_disease}_model.pkl')
scaler = joblib.load(MODEL_DIR/f'{bundle_disease}_scaler.pkl')
encoder= joblib.load(MODEL_DIR/f'{bundle_disease}_encoder.pkl')
feats  = encoder['__features__']
X_tr, X_te, y_tr, y_te, _ = splits[bundle_disease]
sample = X_te[0:1]                         # already scaled
print(f"Disease={bundle_disease}  prediction={model.predict(sample)[0]}  actual={y_te[0]}")